# Etapa 3 — Primer smoke test neuronal cíclico

Esta es la primera prueba en la que el modelo recibe ventanas y **no** las fases verdaderas. Entrenamos un encoder online, un target encoder EMA y un predictor lineal de dimensión 3 sobre la dinámica cíclica. La loss usa predicción más centrado, varianza y decorrelación genéricos; no contiene la matriz cíclica ni sus eigenvalues.

El encoder es convolucional y position-aware: preserva la posición temporal porque en este dataset las cuatro fases son traslaciones del mismo template. El global average pooling de Fase 0 eliminaría justamente esa señal y se reservará como control posterior.

Alcance: una seed, train/validation con realizaciones distintas y checkpoint elegido por mínima validation loss total. Test no se construye ni consulta. Este es un smoke test de mecanismo, no el resultado multi-seed final.

In [ ]:
# ruff: noqa: E402, E501
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import yaml
from IPython.display import Markdown, display
from sklearn.decomposition import PCA

from koopman_jepa.config import DataConfig, ExperimentConfig, ModelConfig, TrainConfig, validate_config
from koopman_jepa.koopman import expected_active_spectrum
from koopman_jepa.model import TemporalJEPA
from koopman_jepa.phase_analysis import evaluate_phase_representation
from koopman_jepa.phase_data import PhaseWindowConfig, make_phase_tensor_dataset_splits
from koopman_jepa.training import (
    collect_embeddings,
    evaluate_model_loss,
    select_device,
    set_seed,
    train_model_with_validation_checkpoint,
)

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
CONFIG_PATH = ROOT / "configs" / "stage3_cyclic_neural_smoke.yaml"
with CONFIG_PATH.open(encoding="utf-8") as handle:
    raw = yaml.safe_load(handle)

dynamics = raw["dynamics"]
base_seed = int(raw["base_seed"])
emission_config = PhaseWindowConfig(**raw["emission"], repeats_per_transition=1)
model_raw = raw["model"]
train_raw = raw["train"]
experiment_config = ExperimentConfig(
    data=DataConfig(context_length=emission_config.window_length),
    model=ModelConfig(
        latent_dim=model_raw["latent_dim"],
        channels=model_raw["channels"],
        predictor_init=model_raw["predictor_init"],
    ),
    train=TrainConfig(seed=base_seed, **train_raw),
)
validate_config(experiment_config)
assert raw["selection"] == {"metric": "validation_total_loss", "mode": "min"}
assert model_raw["pooling"] == "flatten"

splits = make_phase_tensor_dataset_splits(
    emission_config,
    train_repeats_per_transition=raw["splits"]["train_repeats_per_transition"],
    validation_repeats_per_transition=raw["splits"]["validation_repeats_per_transition"],
    seed=base_seed,
)
train_dataset = splits.train[dynamics]
validation_dataset = splits.validation[dynamics]
print(json.dumps({
    "dynamics": dynamics,
    "train_seed": splits.train_seed,
    "validation_seed": splits.validation_seed,
    "train_pairs": len(train_dataset),
    "validation_pairs": len(validation_dataset),
    "test_constructed": False,
}, indent=2))

In [ ]:
set_seed(base_seed)
device = select_device(experiment_config.train.device)
model = TemporalJEPA(
    latent_dim=model_raw["latent_dim"],
    channels=model_raw["channels"],
    predictor_init=model_raw["predictor_init"],
    pooling=model_raw["pooling"],
    input_length=emission_config.window_length,
).to(device)
baseline = evaluate_model_loss(model, validation_dataset, experiment_config, device)
training_result = train_model_with_validation_checkpoint(
    model,
    train_dataset,
    validation_dataset,
    experiment_config,
    device,
)
selected = evaluate_model_loss(model, validation_dataset, experiment_config, device)
history = training_result.history
print(json.dumps({
    "device": str(device),
    "baseline_validation_loss": baseline.loss,
    "selected_epoch": training_result.best_epoch,
    "selected_validation_loss": selected.loss,
    "selected_validation_prediction_loss": selected.prediction_loss,
}, indent=2))

In [ ]:
train_embeddings, _, train_phase_pairs = collect_embeddings(
    model, train_dataset, experiment_config.train.batch_size, device
)
validation_embeddings, _, validation_phase_pairs = collect_embeddings(
    model, validation_dataset, experiment_config.train.batch_size, device
)
predictor_matrix = model.predictor.matrix.detach().cpu().numpy()
metrics = evaluate_phase_representation(
    train_embeddings=train_embeddings,
    train_phases=train_phase_pairs[:, 0],
    validation_embeddings=validation_embeddings,
    validation_phases=validation_phase_pairs[:, 0],
    predictor_matrix=predictor_matrix,
    dynamics=dynamics,
    seed=base_seed,
)
validation_loss_ratio = selected.loss / baseline.loss
history_is_finite = all(np.isfinite(value) for row in history for value in row.values())
summary = {
    "selected_epoch": training_result.best_epoch,
    "validation_loss_ratio": float(validation_loss_ratio),
    "history_is_finite": bool(history_is_finite),
    **metrics,
}
print(json.dumps(summary, indent=2))

In [ ]:
gate_config = raw["gates"]
gate_checks = {
    "finite_training": history_is_finite,
    "validation_improves": validation_loss_ratio <= gate_config["maximum_validation_loss_ratio"],
    "embedding_scale": metrics["embedding_std_mean"] >= gate_config["minimum_embedding_std_mean"],
    "effective_rank": metrics["effective_rank"] >= gate_config["minimum_effective_rank"],
    "phase_probe": metrics["linear_probe_accuracy"] >= gate_config["minimum_linear_probe_accuracy"],
    "phase_alignment": metrics["phase_alignment_error"] <= gate_config["maximum_phase_alignment_error"],
    "active_rank": metrics["active_rank"] == gate_config["required_active_rank"],
    "intertwining": metrics["intertwining_error"] <= gate_config["maximum_intertwining_error"],
    "active_invariance": metrics["active_invariance_error"] is not None and metrics["active_invariance_error"] <= gate_config["maximum_active_invariance_error"],
    "spectrum": metrics["spectral_max_absolute_error"] is not None and metrics["spectral_max_absolute_error"] <= gate_config["maximum_spectral_error"],
}
smoke_gate_passed = bool(all(gate_checks.values()))
print(json.dumps({**gate_checks, "smoke_gate_passed": smoke_gate_passed}, indent=2))
assert smoke_gate_passed

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11), constrained_layout=True)
epochs = np.array([row["epoch"] for row in history])
axes[0, 0].plot(epochs, [row["train_loss"] for row in history], label="train total")
axes[0, 0].plot(epochs, [row["val_loss"] for row in history], label="validation total")
axes[0, 0].plot(epochs, [row["val_prediction_loss"] for row in history], label="validation prediction")
axes[0, 0].axvline(training_result.best_epoch, color="black", linestyle=":", label="checkpoint")
axes[0, 0].set(title="Curvas de entrenamiento", xlabel="Época", ylabel="Loss", yscale="log")
axes[0, 0].legend()

projection = PCA(n_components=2).fit_transform(validation_embeddings)
scatter = axes[0, 1].scatter(projection[:, 0], projection[:, 1], c=validation_phase_pairs[:, 0], cmap="tab10", s=18, alpha=0.65)
axes[0, 1].set(title="Latent de validation", xlabel="PC1", ylabel="PC2")
fig.colorbar(scatter, ax=axes[0, 1], ticks=np.arange(4), label="Fase actual")

covariance_eigenvalues = np.array(metrics["covariance_eigenvalues"])
axes[1, 0].bar(np.arange(len(covariance_eigenvalues)), covariance_eigenvalues)
axes[1, 0].set(title=f"Covarianza latent (rango efectivo {metrics['effective_rank']:.2f})", xlabel="Componente", ylabel="Eigenvalue", xticks=np.arange(len(covariance_eigenvalues)))

estimated_spectrum = np.linalg.eigvals(predictor_matrix)
expected_spectrum = expected_active_spectrum(dynamics, emission_config.num_phases)
axes[1, 1].scatter(expected_spectrum.real, expected_spectrum.imag, marker="x", s=140, label="esperado")
axes[1, 1].scatter(estimated_spectrum.real, estimated_spectrum.imag, facecolors="none", edgecolors="tab:orange", s=80, label="aprendido")
axes[1, 1].add_patch(plt.Circle((0, 0), 1, fill=False, linestyle=":", color="gray"))
axes[1, 1].axhline(0, color="black", linewidth=0.5)
axes[1, 1].axvline(0, color="black", linewidth=0.5)
axes[1, 1].set(title="Espectro del predictor", xlabel="Real", ylabel="Imaginaria", aspect="equal", xlim=(-1.3, 1.3), ylim=(-1.3, 1.3))
axes[1, 1].legend()
plt.show()

display(Markdown(f"""## Análisis del resultado

- Gate del smoke neuronal cíclico: **{'PASS' if smoke_gate_passed else 'FAIL'}**.
- Checkpoint seleccionado: época **{training_result.best_epoch}** de {experiment_config.train.epochs}; ratio validation/baseline: **{validation_loss_ratio:.3f}**.
- Escala media / rango efectivo: **{metrics['embedding_std_mean']:.3f} / {metrics['effective_rank']:.3f}**.
- Accuracy del linear probe de fase: **{metrics['linear_probe_accuracy']:.2%}**.
- Error de alineación con las indicadoras centradas: **{metrics['phase_alignment_error']:.3f}**.
- Rango del subespacio de fase: **{metrics['active_rank']}**.
- Error de entrelazamiento / invariancia activa: **{metrics['intertwining_error']:.3f} / {metrics['active_invariance_error']:.3f}**.
- Error espectral medio / máximo: **{metrics['spectral_mean_absolute_error']:.3f} / {metrics['spectral_max_absolute_error']:.3f}**.

El probe y el gráfico PCA miden si el encoder conserva información de fase; no bastan para afirmar Koopman. El gate exige además que el predictor cierre ese subespacio y tenga eigenvalues cercanos a `−1`, `i` y `−i`. Por eso una representación separable puede todavía producir `FAIL`.

Este resultado usa una sola seed y sólo validation. Un PASS habilitará congelar el protocolo común para las tres dinámicas y varias seeds; un FAIL se diagnosticará por el primer bloque que no pase, sin consultar test ni cambiar retrospectivamente estos thresholds.
"""))